In [1]:
import os
import re
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
from diffusers import UNet2DConditionModel, DDPMScheduler, AutoencoderKL, StableDiffusionPipeline
from transformers import CLIPTextModel, CLIPTokenizer
from accelerate import Accelerator
from torch.optim.lr_scheduler import CosineAnnealingLR
import math

The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.


0it [00:00, ?it/s]

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
class CustomImageDataset(Dataset):
    def __init__(self, image_dir, img_size=(512, 512), augment=True):
        self.image_dir = image_dir
        self.img_size = img_size
        self.augment = augment
        self.image_paths = [
            os.path.join(image_dir, filename)
            for filename in os.listdir(image_dir)
            if filename.lower().endswith(('.jpeg', '.jpg', '.png'))
        ]
        self.transform = self.get_transforms()

    def get_transforms(self):
        transform_list = [
            transforms.Resize(self.img_size),
            transforms.CenterCrop(self.img_size),
        ]
        if self.augment:
            transform_list.extend([
                transforms.RandomHorizontalFlip(),
                transforms.RandomVerticalFlip(),
                transforms.RandomRotation(15),  # Rotate by ±15 degrees
                transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1, hue=0.05),
            ])
        transform_list.extend([
            transforms.ToTensor(),
            transforms.Normalize([0.5], [0.5]),
        ])
        return transforms.Compose(transform_list)

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        image = Image.open(img_path).convert('RGB')
        image = self.transform(image)
        return image

# Set up the dataset and dataloader
image_dir = '/content/drive/MyDrive/Spare/img_cropped'
dataset = CustomImageDataset(image_dir)
dataloader = DataLoader(dataset, batch_size=4, shuffle=True, num_workers=4)

In [12]:
# specify a checkpoint to resume from
resume_from_checkpoint = None  # Replace with checkpoint path if resuming


accelerator = Accelerator(mixed_precision='fp16' if torch.cuda.is_available() else 'no')
device = accelerator.device

# Load the pre-trained model
pretrained_model_name_or_path = "runwayml/stable-diffusion-v1-5"

# Load tokenizer and text encoder
tokenizer = CLIPTokenizer.from_pretrained(pretrained_model_name_or_path, subfolder="tokenizer")
text_encoder = CLIPTextModel.from_pretrained(pretrained_model_name_or_path, subfolder="text_encoder")

# Load UNet and VAE models
if resume_from_checkpoint is not None:
    # Load the UNet model from the checkpoint
    unet = UNet2DConditionModel.from_pretrained(os.path.join(resume_from_checkpoint, 'unet'))
else:
    unet = UNet2DConditionModel.from_pretrained(pretrained_model_name_or_path, subfolder="unet")

vae = AutoencoderKL.from_pretrained(pretrained_model_name_or_path, subfolder="vae")

# Set up the noise scheduler
noise_scheduler = DDPMScheduler.from_pretrained(pretrained_model_name_or_path, subfolder="scheduler")

# Set models to evaluation mode
vae.eval()
text_encoder.eval()


vae.to(device)
text_encoder.to(device)


# Set up the optimizer
optimizer = torch.optim.AdamW(unet.parameters(), lr=1e-5)


/usr/local/lib/python3.10/dist-packages/accelerate/accelerator.py:494: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)
/usr/local/lib/python3.10/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


unet/config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

diffusion_pytorch_model.safetensors:   0%|          | 0.00/3.44G [00:00<?, ?B/s]

scheduler/scheduler_config.json:   0%|          | 0.00/308 [00:00<?, ?B/s]

In [6]:
num_epochs = 15
num_training_steps = num_epochs * len(dataloader)
lr_scheduler = CosineAnnealingLR(optimizer, T_max=num_training_steps)
gradient_accumulation_steps = 1

# Prepare models and dataloader
unet, optimizer, dataloader, lr_scheduler = accelerator.prepare(
    unet, optimizer, dataloader, lr_scheduler
)


start_epoch = 0

# Load training state if resuming
if resume_from_checkpoint is not None:
    accelerator.load_state(resume_from_checkpoint)
    # Extract the epoch number from the checkpoint path (assuming the naming convention 'checkpoint_epoch_{epoch}')
    match = re.search(r'checkpoint_epoch_(\d+)', resume_from_checkpoint)
    if match is not None:
        start_epoch = int(match.group(1)) + 1
    else:
        start_epoch = 0

# Training loop
for epoch in range(start_epoch, num_epochs):
    unet.train()  # Set UNet to training mode
    for step, batch in enumerate(dataloader):
        with accelerator.accumulate(unet):
            # Move batch to device
            batch = batch.to(device)

            # Encode images to latent space
            with torch.no_grad():
                latents = vae.encode(batch).latent_dist.sample() * 0.18215  # Scaling factor

            # Sample random noise and timestep
            noise = torch.randn_like(latents)
            timesteps = torch.randint(
                0, noise_scheduler.config.num_train_timesteps, (latents.shape[0],), device=device
            ).long()

            # Add noise to the latents
            noisy_latents = noise_scheduler.add_noise(latents, noise, timesteps)

            # Prepare text embeddings (using a fixed prompt for all images)
            prompt = "a used teabag"  # Replace with your object description
            text_input = tokenizer(
                prompt,
                padding="max_length",
                max_length=tokenizer.model_max_length,
                truncation=True,
                return_tensors="pt",
            )
            with torch.no_grad():
                text_embeddings = text_encoder(text_input.input_ids.to(device))[0]

            # Repeat text_embeddings to match batch size
            batch_size = noisy_latents.shape[0]
            text_embeddings = text_embeddings.repeat(batch_size, 1, 1)

            # Predict the noise residual
            noise_pred = unet(noisy_latents, timesteps, encoder_hidden_states=text_embeddings).sample

            # Compute loss
            loss = torch.nn.functional.mse_loss(noise_pred, noise, reduction="mean")

            # Backpropagation
            accelerator.backward(loss)

            # Update weights and optimizer
            if accelerator.sync_gradients:
                optimizer.step()
                lr_scheduler.step()
                optimizer.zero_grad()

            # Logging
            if step % 10 == 0:
                current_lr = optimizer.param_groups[0]['lr']
                print(f"Epoch {epoch}, Step {step}, Loss: {loss.item():.4f}, LR: {current_lr:.6f}")

    # Save the state and model after each epoch
    accelerator.wait_for_everyone()
    if accelerator.is_main_process:
        save_path = f"checkpoint_epoch_{epoch}"
        accelerator.save_state(save_path)
        # Save the unwrapped UNet model for inference
        unet_to_save = accelerator.unwrap_model(unet)
        unet_to_save.save_pretrained(os.path.join(save_path, 'unet'))

Epoch 0, Step 0, Loss: 0.1335, LR: 0.000010
Epoch 0, Step 10, Loss: 0.1521, LR: 0.000010
Epoch 0, Step 20, Loss: 0.0304, LR: 0.000010
Epoch 0, Step 30, Loss: 0.0683, LR: 0.000010
Epoch 0, Step 40, Loss: 0.0637, LR: 0.000010
Epoch 0, Step 50, Loss: 0.2366, LR: 0.000010
Epoch 0, Step 60, Loss: 0.1051, LR: 0.000010
Epoch 0, Step 70, Loss: 0.0204, LR: 0.000010
Epoch 0, Step 80, Loss: 0.0698, LR: 0.000010
Epoch 0, Step 90, Loss: 0.0584, LR: 0.000010
Epoch 0, Step 100, Loss: 0.0102, LR: 0.000010
Epoch 0, Step 110, Loss: 0.0547, LR: 0.000010
Epoch 0, Step 120, Loss: 0.1312, LR: 0.000010
Epoch 0, Step 130, Loss: 0.0602, LR: 0.000010
Epoch 0, Step 140, Loss: 0.1163, LR: 0.000010
Epoch 0, Step 150, Loss: 0.0852, LR: 0.000010
Epoch 1, Step 0, Loss: 0.0809, LR: 0.000010
Epoch 1, Step 10, Loss: 0.0508, LR: 0.000010
Epoch 1, Step 20, Loss: 0.0431, LR: 0.000010
Epoch 1, Step 30, Loss: 0.0308, LR: 0.000010
Epoch 1, Step 40, Loss: 0.0848, LR: 0.000010
Epoch 1, Step 50, Loss: 0.1064, LR: 0.000010
Epoch 

Image generation


In [25]:
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
torch.cuda.empty_cache()


In [5]:
!cp /content/drive/My\ Drive/epoch_14.zip /content

In [6]:
!unzip /content/epoch_14.zip

Archive:  /content/epoch_14.zip
   creating: content/checkpoint_epoch_14/
  inflating: content/checkpoint_epoch_14/scheduler.bin  
  inflating: content/checkpoint_epoch_14/scaler.pt  
  inflating: content/checkpoint_epoch_14/random_states_0.pkl  
  inflating: content/checkpoint_epoch_14/optimizer.bin  
  inflating: content/checkpoint_epoch_14/model.safetensors  
   creating: content/checkpoint_epoch_14/unet/
  inflating: content/checkpoint_epoch_14/unet/diffusion_pytorch_model.safetensors  
  inflating: content/checkpoint_epoch_14/unet/config.json  


In [ ]:
# Specify the path to your checkpoint
checkpoint_path = "/content/content/checkpoint_epoch_14"  # Replace with your actual checkpoint path

# Load the pre-trained models
pretrained_model_name_or_path = "runwayml/stable-diffusion-v1-5"

# Load tokenizer and text encoder
tokenizer = CLIPTokenizer.from_pretrained(
    pretrained_model_name_or_path, subfolder="tokenizer"
)
text_encoder = CLIPTextModel.from_pretrained(
    pretrained_model_name_or_path, subfolder="text_encoder"
)

# Load VAE model
vae = AutoencoderKL.from_pretrained(
    pretrained_model_name_or_path, subfolder="vae"
)

# Load the fine-tuned UNet model from the checkpoint
unet = UNet2DConditionModel.from_pretrained(
    checkpoint_path, subfolder="unet"
)

# Load the scheduler
scheduler = DDPMScheduler.from_pretrained(
    pretrained_model_name_or_path, subfolder="scheduler"
)

# Move models to the appropriate device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
vae.to(device)
text_encoder.to(device)
unet.to(device)

# Create the Stable Diffusion pipeline
pipeline = StableDiffusionPipeline(
    vae=vae,
    text_encoder=text_encoder,
    tokenizer=tokenizer,
    unet=unet,
    scheduler=scheduler,
    safety_checker=None,
    feature_extractor=None,
).to(device)



In [26]:

generator = torch.Generator(device=device).manual_seed(0)

prompt = "a used teabag"

# Generate images
with torch.autocast(device.type):
    images = pipeline(
        prompt,
        num_inference_steps=50,
        guidance_scale=7.5,
        generator=generator,
        num_images_per_prompt=10,
    ).images

# Save generated images
for idx, image in enumerate(images):
    image.save(f"/content/gen_e13/generated_image_{idx}.png")

  0%|          | 0/50 [00:00<?, ?it/s]

In [28]:
!zip -r 2.zip /content/gen_e13

  adding: content/gen_e13/ (stored 0%)
  adding: content/gen_e13/generated_image_5.png (deflated 0%)
  adding: content/gen_e13/generated_image_6.png (deflated 0%)
  adding: content/gen_e13/generated_image_9.png (deflated 0%)
  adding: content/gen_e13/generated_image_4.png (deflated 0%)
  adding: content/gen_e13/generated_image_8.png (deflated 0%)
  adding: content/gen_e13/generated_image_1.png (deflated 0%)
  adding: content/gen_e13/generated_image_7.png (deflated 0%)
  adding: content/gen_e13/generated_image_2.png (deflated 0%)
  adding: content/gen_e13/generated_image_0.png (deflated 0%)
  adding: content/gen_e13/generated_image_3.png (deflated 0%)


In [20]:
import locale

def getpreferredencoding(do_setlocale = True):
    return "UTF-8"

locale.getpreferredencoding = getpreferredencoding

In [53]:
!zip -r /content/epoch_14.zip /content/checkpoint_epoch_14


  adding: content/checkpoint_epoch_14/ (stored 0%)
  adding: content/checkpoint_epoch_14/scheduler.bin (deflated 57%)
  adding: content/checkpoint_epoch_14/scaler.pt (deflated 60%)
  adding: content/checkpoint_epoch_14/random_states_0.pkl (deflated 25%)
  adding: content/checkpoint_epoch_14/optimizer.bin (deflated 9%)
  adding: content/checkpoint_epoch_14/model.safetensors (deflated 7%)
  adding: content/checkpoint_epoch_14/unet/ (stored 0%)
  adding: content/checkpoint_epoch_14/unet/diffusion_pytorch_model.safetensors (deflated 7%)
  adding: content/checkpoint_epoch_14/unet/config.json (deflated 66%)
